set the fibril source paths and load any existing download records

In [1]:
import hashlib
import json
import math
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

import gemmi
from bs4 import BeautifulSoup

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

raw_root = project_root / "data" / "raw" / "alpha_synuclein_fibrils"
raw_root.mkdir(parents=True, exist_ok=True)
manifest_path = project_root / "data" / "fibril_manifest.json"
fibril_manifest = (
    json.loads(manifest_path.read_text())
    if manifest_path.exists()
    else {"files": {}, "failures": {}}
)
fibril_manifest["dataset_version"] = None

download source files with checksums and reuse only verified local copies

In [2]:
def save_fibril_manifest():
    pending_path = manifest_path.with_suffix(".json.part")
    pending_path.write_text(json.dumps(fibril_manifest, indent=2, sort_keys=True) + "\n")
    pending_path.replace(manifest_path)


def download_fibril_source(url, path):
    relative_path = path.relative_to(project_root).as_posix()
    previous = fibril_manifest["files"].get(relative_path)
    if previous and path.exists():
        if previous["url"] != url or hashlib.sha256(path.read_bytes()).hexdigest() != previous["sha256"]:
            raise ValueError(f"source or checksum changed: {relative_path}")
        return previous

    for attempt in range(3):
        try:
            request = Request(url, headers={"User-Agent": "reaSyn/0.1 (https://github.com/appliedCL/reaSyn)"})
            with urlopen(request, timeout=60) as response:
                content = response.read()
            break
        except (HTTPError, URLError, TimeoutError) as error:
            if attempt == 2 or isinstance(error, HTTPError) and error.code == 404:
                raise
            time.sleep(2 ** attempt)

    path.parent.mkdir(parents=True, exist_ok=True)
    pending_path = path.with_suffix(path.suffix + ".part")
    pending_path.write_bytes(content)
    pending_path.replace(path)
    return {
        "url": url,
        "retrieved_at": datetime.now(timezone.utc).isoformat(),
        "sha256": hashlib.sha256(content).hexdigest(),
        "bytes": len(content),
    }


def download_fibril_batch(sources):
    with ThreadPoolExecutor(max_workers=4) as pool:
        jobs = {pool.submit(download_fibril_source, url, path): (url, path) for url, path in sources}
        for completed, job in enumerate(as_completed(jobs), 1):
            url, path = jobs[job]
            relative_path = path.relative_to(project_root).as_posix()
            try:
                fibril_manifest["files"][relative_path] = job.result()
                fibril_manifest["failures"].pop(relative_path, None)
            except Exception as error:
                fibril_manifest["failures"][relative_path] = {"url": url, "error": str(error)}
            save_fibril_manifest()
            if completed % 50 == 0 or completed == len(jobs):
                print(f"{completed}/{len(jobs)} source files checked", flush=True)

snapshot the atlas and retain every alpha-synuclein entry with its selection reason

In [3]:
atlas_url = "https://people.mbi.ucla.edu/sawaya/amyloidatlas/"
atlas_path = raw_root / "amyloid_atlas.html"
fibril_manifest["files"][atlas_path.relative_to(project_root).as_posix()] = download_fibril_source(
    atlas_url, atlas_path
)
atlas_cells = BeautifulSoup(atlas_path.read_bytes(), "html.parser").select("table td")
assert len(atlas_cells) % 9 == 0, "atlas column layout has changed"
assert atlas_cells[0].get_text(strip=True) == "Protein"
assert atlas_cells[7].get_text(strip=True) == "PDB ID"

atlas_entries = []
for offset in range(9, len(atlas_cells), 9):
    cells = atlas_cells[offset:offset + 9]
    values = [cell.get_text(" ", strip=True) for cell in cells]
    if values[0] != "α-synuclein":
        continue
    reference = cells[8].find("a", href=True)
    atlas_entries.append({
        "pdb_id": values[7].upper(),
        "protein": values[0],
        "origins_text": values[1],
        "ordered_residues_text": values[2],
        "method": values[3],
        "resolution_text": values[4],
        "reference_text": values[8],
        "reference_url": reference["href"] if reference else None,
        "selected": values[3] == "cryoEM",
        "selection_reason": "alpha-synuclein cryo-EM" if values[3] == "cryoEM" else "outside cryo-EM scope",
    })

pdb_ids = sorted(entry["pdb_id"] for entry in atlas_entries if entry["selected"])
assert pdb_ids and len(pdb_ids) == len(set(pdb_ids)), "missing or duplicate PDB identifiers"
assert all(re.fullmatch(r"[1-9][A-Z0-9]{3}", pdb_id) for pdb_id in pdb_ids)
fibril_manifest["atlas_entries"] = atlas_entries
save_fibril_manifest()
print(f"{len(pdb_ids)} cryo-EM entries selected; {len(atlas_entries) - len(pdb_ids)} other-method entries recorded")

225 cryo-EM entries selected; 2 other-method entries recorded


download the coordinates and pdb entry metadata for each selected structure

In [4]:
pdb_sources = [
    source
    for pdb_id in pdb_ids
    for source in (
        (f"https://files.rcsb.org/download/{pdb_id}.cif", raw_root / "coordinates" / f"{pdb_id}.cif"),
        (f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}", raw_root / "pdb_metadata" / f"{pdb_id}.json"),
    )
]
download_fibril_batch(pdb_sources)

50/450 source files checked
100/450 source files checked
150/450 source files checked
200/450 source files checked
250/450 source files checked
300/450 source files checked
350/450 source files checked
400/450 source files checked
450/450 source files checked


download metadata for the unique emdb maps linked by the pdb entries

In [5]:
emdb_ids = set()
for pdb_id in pdb_ids:
    metadata_path = raw_root / "pdb_metadata" / f"{pdb_id}.json"
    if metadata_path.relative_to(project_root).as_posix() in fibril_manifest["failures"]:
        continue
    metadata = json.loads(metadata_path.read_text())
    assert metadata["rcsb_id"] == pdb_id
    emdb_ids.update(metadata.get("rcsb_entry_container_identifiers", {}).get("emdb_ids", []))

emdb_sources = [
    (f"https://www.ebi.ac.uk/emdb/api/entry/{emdb_id}", raw_root / "emdb_metadata" / f"{emdb_id}.json")
    for emdb_id in sorted(emdb_ids)
]
download_fibril_batch(emdb_sources)

50/219 source files checked
100/219 source files checked
150/219 source files checked
200/219 source files checked
219/219 source files checked


check coordinate parsing and flag identity or method discrepancies without dropping entries

In [6]:
structure_checks = {}
for pdb_id in pdb_ids:
    coordinate_path = raw_root / "coordinates" / f"{pdb_id}.cif"
    metadata_path = raw_root / "pdb_metadata" / f"{pdb_id}.json"
    flags = []
    try:
        block = gemmi.cif.read_file(str(coordinate_path)).sole_block()
        assert block.find_value("_entry.id").upper() == pdb_id
        positions = block.find(["_atom_site.Cartn_x", "_atom_site.Cartn_y", "_atom_site.Cartn_z"])
        assert len(positions) > 0
        assert all(math.isfinite(float(value)) for row in positions for value in row)
        descriptions = list(block.find_values("_entity.pdbx_description"))
        if not any(re.search(r"(alpha|α)[ -]?synuclein", text, re.IGNORECASE) for text in descriptions):
            flags.append("protein identity needs review")
        metadata = json.loads(metadata_path.read_text())
        methods = [method["method"] for method in metadata.get("exptl", [])]
        if "ELECTRON MICROSCOPY" not in methods:
            flags.append("experimental method needs review")
        if metadata.get("em_experiment", {}).get("aggregation_state") != "FILAMENT":
            flags.append("fibril state needs review")
        structure_checks[pdb_id] = {"atom_count": len(positions), "review_flags": flags}
    except Exception as error:
        structure_checks[pdb_id] = {"atom_count": None, "review_flags": [f"validation failed: {error}"]}

for emdb_id in sorted(emdb_ids):
    path = raw_root / "emdb_metadata" / f"{emdb_id}.json"
    if path.relative_to(project_root).as_posix() not in fibril_manifest["failures"]:
        assert isinstance(json.loads(path.read_text()), dict)

fibril_manifest["structure_checks"] = structure_checks
fibril_manifest["dataset_version"] = hashlib.sha256(
    "\n".join(
        f"{path}:{record['sha256']}"
        for path, record in sorted(fibril_manifest["files"].items())
    ).encode()
).hexdigest()
save_fibril_manifest()

summary = {
    "selected_structures": len(pdb_ids),
    "parsed_structures": sum(check["atom_count"] is not None for check in structure_checks.values()),
    "linked_emdb_entries": len(emdb_ids),
    "downloaded_files": len(fibril_manifest["files"]),
    "downloaded_bytes": sum(record["bytes"] for record in fibril_manifest["files"].values()),
    "download_failures": fibril_manifest["failures"],
    "structures_for_review": {pdb_id: check["review_flags"] for pdb_id, check in structure_checks.items() if check["review_flags"]},
    "dataset_version": fibril_manifest["dataset_version"],
}
print(json.dumps(summary, indent=2))
assert not fibril_manifest["failures"], "download failures are recorded in the manifest"
assert summary["parsed_structures"] == len(pdb_ids), "coordinate validation failures need review"

{
  "selected_structures": 225,
  "parsed_structures": 225,
  "linked_emdb_entries": 219,
  "downloaded_files": 670,
  "downloaded_bytes": 122110299,
  "download_failures": {},
  "structures_for_review": {
    "8X7B": [
      "fibril state needs review"
    ],
    "8X7L": [
      "fibril state needs review"
    ],
    "8X7M": [
      "fibril state needs review"
    ],
    "8X7O": [
      "fibril state needs review"
    ],
    "8X7P": [
      "fibril state needs review"
    ],
    "8X7Q": [
      "fibril state needs review"
    ],
    "8X7R": [
      "fibril state needs review"
    ],
    "8ZLI": [
      "fibril state needs review"
    ],
    "8ZLO": [
      "fibril state needs review"
    ],
    "8ZLP": [
      "fibril state needs review"
    ],
    "8ZMY": [
      "fibril state needs review"
    ],
    "9KA3": [
      "fibril state needs review"
    ],
    "9KAL": [
      "fibril state needs review"
    ]
  },
  "dataset_version": "092458e487fed389912256a345645526dff1251729f5eb8bba1f2